# Partitioning

**What this cell does:** Hides Python warnings so the output stays readable.

**Core concept:** This only affects Python-side warnings in the notebook (for example deprecation notices). It has no effect on Spark execution or Spark's own logs.

**What this cell does:** Makes the notebook display the result of every bare expression in a cell, not only the last one.

**Core concept:** By default a cell only shows the value of its last line. With `ast_node_interactivity = 'all'`, lines such as `df.count()` in the middle of a cell also print their result. This is a Jupyter/IPython setting and also works in Databricks Python notebooks.

**What this cell does:** Imports Spark types, the functions module (`F`) and `StorageLevel`.

**Core concept:** On Databricks the `SparkSession` is created for you and exposed as `spark`, so there is no need to import or build it. `F` gives access to built-in column functions such as `F.to_date`, `F.hour` and `F.spark_partition_id`.

**What this cell does:** Defines the input and output folders in the Unity Catalog Volume.

**Core concept:** In local Jupyter the original code built a Spark session with `master("local[*]")` and `spark.driver.memory`. On Databricks the cluster (or serverless) provides the session and those resources, so the notebook only needs paths. Volumes (`/Volumes/<catalog>/<schema>/<volume>/...`) are governed file storage backed by cloud storage (ADLS here).

**What this cell does:** Reads `Spotify_Listening_Activity.csv` into a DataFrame and shows the first 5 rows.

**Core concept:** `header=True` uses the first line as column names; `inferSchema=True` makes Spark scan the data to guess column types (so `listen_date` becomes a timestamp and IDs become integers). Schema inference costs an extra pass over the file; for large data an explicit schema is faster. `show(5, False)` prints rows without truncating long values.

**What this cell does:** Renames the timestamp column to `listen_time` and derives two new columns, `listen_date` (date only) and `listen_hour` (0-23), then shows the result, the schema and the row count (11,779 rows).

**Core concept:** These are narrow transformations: each row is computed independently, so no data moves between partitions. The derived columns are created so the data can later be partitioned (laid out in folders) by date and hour. A good partition column has a manageable number of distinct values and matches how the data is usually filtered.

**What this cell does:** Writes the data as parquet into one subfolder per date, for example `listening_activity_pt/listen_date=2023-06-27/part-....parquet`.

**Core concept:** `partitionBy` controls the **layout on disk**, not the number of in-memory partitions. The partition value is stored in the folder name and removed from the data files. `mode("overwrite")` replaces the whole output folder each run. Each write task produces a separate file in every date folder it has rows for, so the number of files per folder depends on how many in-memory partitions contain that date.

**What this cell does:** Reads the partitioned folder back, prints the physical plan for a filter on `listen_date = '2019-01-01'`, then displays the rows for `2023-06-27`.

**Core concept:** **Partition pruning**. Because `listen_date` is a partition column, Spark compares the filter against folder names and only opens files in matching folders. In the plan this appears as `PartitionFilters: [... (listen_date = 2019-01-01)]`, with empty `DataFilters`, and `listen_date` is missing from `ReadSchema` because it comes from the folder name. The data is from 2023, so the 2019 filter matches no folder and reads 0 files; the 2023-06-27 filter reads only that one folder. `explain()` only prints the plan; `display()` actually runs the query.

**What this cell does:** Partitions by two columns, producing nested folders `listen_date=<date>/listen_hour=<hour>/`.

**Core concept:** Multi-level partitioning lets Spark prune on either level (date, or date and hour). The order of the columns sets the folder hierarchy: the first column is the top level. Every extra level multiplies the number of folders and files, so too many levels or high-cardinality columns create many small files, which slows down listing and reading.

**What this cell does:** Same as above with the order reversed: `listen_hour=<hour>/listen_date=<date>/`.

**Core concept:** The total number of leaf folders is the same, but the hierarchy is different. Choose the order that matches the most common filter: put the column you filter on most often (usually the coarser one, like date) first, so a filter on it prunes whole top-level folders.

**What this cell does:** Redistributes the data into 3 in-memory partitions with `repartition(3)`, then writes it partitioned by `listen_date`.

**Core concept:** `repartition(n)` always performs a full shuffle and spreads rows evenly (round-robin) over `n` partitions. During the write, each of the 3 tasks writes one file into every date folder it has rows for, so each date folder ends up with **up to 3 files** (verified: all 126 date folders in `listening_activity_pt_4` contain 3 files). This is how you control the number of files per folder: in-memory partitions (`repartition`/`coalesce`) decide files per folder, `partitionBy` decides the folders.

**What this cell does:** Reduces to at most 3 in-memory partitions with `coalesce(3)`, then writes partitioned by `listen_date`.

**Core concept:** `coalesce(n)` merges existing partitions **without a shuffle**, so it can only reduce the count, never increase it. Note: the comment in the cell saying it "shuffles the data using default Hash Partitioner" is not accurate; coalesce does not shuffle. This CSV is read as a single partition (see the `maxPartitionBytes` section below), so `coalesce(3)` keeps 1 partition and each date folder gets 1 file (verified: all 126 folders in `listening_activity_pt_5` contain 1 file), unlike `repartition(3)` above.

**What this cell does:** Saves the current `spark.sql.files.maxPartitionBytes` value (so it can be restored later), reads the CSV and counts its read partitions. Result: 1 partition at the default 128 MB (`134217728b`).

**Core concept:** When Spark reads files, it splits them into read partitions of at most `maxPartitionBytes` each; one partition becomes one task. The whole file is about 445 KB, far below 128 MB, so it fits in a single partition. `df.rdd.getNumPartitions()` is not available on serverless/standard access mode, so the cell counts distinct `spark_partition_id()` values instead (this counts non-empty partitions).

**What this cell does:** Lists the CSV file in the Volume and prints its size (456,056 bytes, about 445 KB).

**Core concept:** The file size is the input to the partition calculation: in open-source Spark, partitions are roughly `file size / maxPartitionBytes`. Knowing the size lets you predict the partition count in the next cell and compare it with what Spark actually does.

**What this cell does:** Sets `maxPartitionBytes` to 1024 bytes (1 KB), reads the same CSV again and counts its partitions.

**Core concept:** A smaller `maxPartitionBytes` cuts the file into more, smaller read partitions, so more tasks run in parallel (with more scheduling overhead per byte). Open-source Spark would create about 446 partitions here (445 KB / 1 KB). On the serverless compute this notebook ran on, the result was **112**, about `file size / (4 x maxPartitionBytes)`; the same 4x pattern was observed at 1000, 2048, 4096 and 8192 bytes. This is Databricks-specific behaviour (not documented as far as we could find), so the exact count can differ on a classic cluster. Values this small are only useful for experiments.

**What this cell does:** Restores `maxPartitionBytes` to the value saved earlier (128 MB) and prints it.

**Core concept:** `spark.conf.set` changes the setting for the whole session, so every later read in this notebook would use 1 KB splits if it were not restored. On Databricks you do not call `spark.stop()`: the session is managed by the cluster, and stopping it would break the notebook.

In [0]:
import warnings
warnings.filterwarnings("ignore")

In [0]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [0]:
from pyspark.storagelevel import StorageLevel
from pyspark.sql.types import *
import pyspark.sql.functions as F
# SparkSession is pre-created by Databricks and available as `spark`

In [0]:
# On Databricks the `spark` session already exists - no builder / getOrCreate needed.
# spark.driver.memory and master("local[*]") are cluster-level settings (Compute > Advanced options > Spark config).

INPUT_PATH = "/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed/data/input"
OUTPUT_PATH = "/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed/data/output"

In [0]:
listening_activity_file = f"{INPUT_PATH}/Spotify_Listening_Activity.csv"
df_listening_actv = spark.read.csv(listening_activity_file, header=True, inferSchema=True)
df_listening_actv.show(5, False)

In [0]:
df_listening_actv = (
    df_listening_actv
    .withColumnRenamed("listen_date", "listen_time")
    .withColumn("listen_date", F.to_date("listen_time", "yyyy-MM-dd HH:mm:ss.SSSSSS"))
    .withColumn("listen_hour", F.hour("listen_time"))
)

df_listening_actv.show(5, False)
df_listening_actv.printSchema()
df_listening_actv.count()

## Partitioning By `listen_date`

Let's say we want to **analyse the listening behaviours of user over time**. If we're given the complete dataset (with no partitions), Spark would scan the whole dataset for finding a particular date (similar to the bookshelf analogy where you would scan the entire bookself for finding a book if it is not organized). Given that our usecase needs analysis by date, partitioning (creating folders) on date would help Spark pin point to the exact folder. This makes searching very easy and Spark doesn't scan the entire dataset.  

In [0]:
# Partitioning listening activity by the listen date
(
    df_listening_actv
    .write
    .partitionBy("listen_date")
    .mode("overwrite")
    .parquet(f"{OUTPUT_PATH}/listening_activity_pt")
)

## Partition Pruning

In [0]:
df_listening_actv_pt_pruned = spark.read.parquet(f"{OUTPUT_PATH}/listening_activity_pt")
df_listening_actv_pt_pruned.filter("listen_date = '2019-01-01'").explain()
df_listening_actv_pt_pruned.filter("listen_date = '2023-06-27'").display()

## What Problems Does Partitioning Solve? 
1. `Fast Search (Query Performance)`: Spark will only process the relevant partition instead of the entire dataset (example above). This greatly reduces I/O and query execution time. 
2. `Parallelism / Resource Utilization`: Each core processes 1 partition; More number of partitions, more is the parallelism; again this does not mean we forcefully increase the number of partitions. Each partition should be `128MB` in size. 


# Partitioning Examples
1. Single/multi level partitioning
2. Using `repartition`/`coalesce` with `partitionBy` (controlling number of files inside each partition): 
    - `parititionBy` affects how data is laid out in the storage and is going to ensure that the output directory is organized into subdirectories based on the `value` given in `partitionBy`.  
    - Number of files in each `value` directory of `partitionBy` depends on the number supplied in the `repartition`/`coalesce`.

#### 1. Single/multi level partitioning

In [0]:
(
    df_listening_actv
    .write
    .mode("overwrite")
    .partitionBy("listen_date", "listen_hour")
    .parquet(f"{OUTPUT_PATH}/listening_activity_pt_2")
)

In [0]:
(
    df_listening_actv
    .write
    .mode("overwrite")
    .partitionBy("listen_hour", "listen_date")
    .parquet(f"{OUTPUT_PATH}/listening_activity_pt_3")
)

#### 2. Using `repartition`/`coalesce` with `partitionBy`

In [0]:
(
    df_listening_actv
    .repartition(3)
    .write
    .mode("overwrite")
    .partitionBy("listen_date")
    .parquet(f"{OUTPUT_PATH}/listening_activity_pt_4")
)

In [0]:
# The coalesce method reduces the number of partitions in a DataFrame. 
# It avoids full shuffle, instead of creating new partitions, it shuffles the data using default Hash Partitioner, 
# and adjusts into existing partitions, this means it can only decrease the number of partitions.

(
    df_listening_actv
    .coalesce(3)
    .write
    .mode("overwrite")
    .partitionBy("listen_date")
    .parquet(f"{OUTPUT_PATH}/listening_activity_pt_5")
)

## Experimenting With `spark.sql.files.maxPartitionBytes`

In [0]:
# No need to stop/recreate the session on Databricks; keep the original value so it can be restored later
original_max_partition_bytes = spark.conf.get("spark.sql.files.maxPartitionBytes")
print(f"Default maxPartitionBytes: {original_max_partition_bytes}")

df_default = spark.read.csv(f"{INPUT_PATH}/Spotify_Listening_Activity.csv", header=True, inferSchema=True)
# df.rdd is not available in Standard access mode / serverless - count non-empty partitions instead
default_partitions = df_default.select(F.spark_partition_id()).distinct().count()
print(f"Number of partitions with default maxPartitionBytes: {default_partitions}")


In [0]:
file_info = dbutils.fs.ls(f"{INPUT_PATH}/Spotify_Listening_Activity.csv")
for f in file_info:
    print(f"Name: {f.name}, Size: {f.size} bytes ({f.size / (1024 ):.2f} KB)")

In [0]:
spark.conf.set("spark.sql.files.maxPartitionBytes", "1024")

df_modified = spark.read.csv(f"{INPUT_PATH}/Spotify_Listening_Activity.csv", header=True, inferSchema=True)
modified_partitions = df_modified.select(F.spark_partition_id()).distinct().count()
print(f"Number of partitions with modified maxPartitionBytes: {modified_partitions}")


In [0]:
# spark.stop() is not needed on Databricks - restore the config changed above instead
spark.conf.set("spark.sql.files.maxPartitionBytes", original_max_partition_bytes)
print(spark.conf.get("spark.sql.files.maxPartitionBytes"))